# Заняття 15 — Оптимізація виконання завдань в Apache Spark

Кожен приклад — 2–3 сусідні клітинки з різними значеннями одного параметра. Порядок роботи:
клітинка → перевірка (число у виводі + вкладка Spark UI) → наступна клітинка → порівняння.
Адреса Spark UI друкується після створення сесії; вона доступна, поки сесія активна.

Кожна клітинка повертає одне перевірне значення (кількість партицій, час, кількість файлів або
компактний план). Spark UI — вторинна ілюстрація.

Датасет: NYC Yellow Taxi, січень 2024 (~2.96 млн рядків, 48 MB parquet) + довідник зон (265 рядків).
Запуск: `uv run jupyter lab` з кореня репозиторію.

## Три рівні конфігурації

| Рівень | Де задається | Що можна |
|---|---|---|
| `spark-defaults.conf` | кластер | дефолти для всіх джобів |
| `spark-submit --conf k=v` / `PYSPARK_SUBMIT_ARGS` | джоб, **до** старту JVM | `spark.executor.*`, `spark.driver.*`, `*.memoryOverhead`, `spark.serializer` |
| `spark.conf.set("k", v)` | жива сесія | тільки `spark.sql.*` і частина рантайму |

Памʼять драйвера задається через `PYSPARK_SUBMIT_ARGS` (до `import`), свідомо тісно (900 MB), для демонстрації spill.
Решта параметрів змінюється через `spark.conf.set(...)` по ходу ноутбука.

In [1]:
import os
# spark.driver.memory не можна задати через .config() у local-режимі (JVM уже підніметься з дефолтом).
# Значення навмисно тісне (900 MB), задане тут, до першого import pyspark, — для розділу 5 (spill).
# "pyspark-shell" наприкінці — обовʼязковий службовий аргумент.
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 900m pyspark-shell"

import time, glob, shutil, math, pathlib
import pandas as pd, numpy as np
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType

_p = pathlib.Path.cwd()
while not (_p / "pyproject.toml").exists() and _p != _p.parent:
    _p = _p.parent
os.chdir(_p)

SRC = "data/source/yellow_tripdata_2024-01.parquet"
ZONES = "data/source/taxi_zone_lookup.parquet"
EVENTS = pathlib.Path("lesson-15-spark-optimization/code/spark-events")
EVENTS.mkdir(parents=True, exist_ok=True)

# noop: виконати обчислення без матеріалізації файлів (щоб зʼявилось у Spark UI)
def noop(dframe):
    dframe.write.mode("overwrite").format("noop").save()

### Параметри цієї сесії

| Параметр | Призначення |
|---|---|
| `PYSPARK_SUBMIT_ARGS="--driver-memory 900m …"` (клітинка вище) | у local-режимі драйвер = executor, і його heap не змінити через `.config()` — лише цією змінною до старту JVM. 900 MB навмисно тісно, для розділу 5 (spill). |
| `master("local[4]")` | Spark локально в цьому ж процесі, **4** потоки-виконавці (= 4 слоти задач). Фіксоване число — щоб кількість task-ів і партицій була однакова при кожному прогоні. `local[*]` узяв би всі ядра. |
| `appName("L15")` | назва застосунку у Spark UI і в event log. |
| `spark.serializer = KryoSerializer` | компактніший і швидший серіалізатор за дефолтний Java — для shuffle, кешу, broadcast. Задається **тільки** при створенні сесії. |
| `spark.eventLog.enabled = true` | писати журнал подій застосунку на диск — для відкриття прогону у **Spark History Server** після завершення сесії (жива UI на `localhost:4040` доступна лише поки сесія активна). |
| `spark.eventLog.dir = …/spark-events` | куди писати журнал. Тека має існувати **до** старту (створена вище). |
| `spark.sql.adaptive.enabled = false` | вимкнути AQE на старті — щоб перші приклади показували стан **«до»**. У Spark 3.2+ AQE увімкнений за замовчуванням; у розділах 2, 6, 7 вмикається назад. |
| `spark.ui.showConsoleProgress = false` | прибрати ASCII-полоски прогресу `[Stage 3:===>]` зі stdout. На виконання не впливає. |
| `.getOrCreate()` | створити сесію або повернути наявну. Повторний виклик у тому ж ядрі ігнорує нові `.config(...)` для незмінних параметрів — зміна `driver.memory` / `serializer` потребує перезапуску ядра. |

In [2]:
spark = (
    SparkSession.builder
    .master("local[4]")                                # локально, 4 слоти задач
    .appName("L15")                                    # назва у Spark UI / event log
    .config("spark.serializer",
            "org.apache.spark.serializer.KryoSerializer")   # компактніший серіалізатор (тільки при старті)
    .config("spark.eventLog.enabled", "true")          # журнал подій → Spark History Server
    .config("spark.eventLog.dir", EVENTS.resolve().as_uri())
    .config("spark.sql.adaptive.enabled", "false")     # AQE OFF на старті — стан "до"
    .config("spark.ui.showConsoleProgress", "false")   # без ASCII-полосок прогресу в stdout
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")     # менше логів; ERROR — ще менше
df = spark.read.parquet(SRC)

print("Spark UI:", spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/09/09 22:01:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/09/09 22:01:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark UI: http://mac:4041


## 1. `spark.sql.shuffle.partitions` — 200 vs 16

Кількість партицій **після shuffle** (`groupBy`, `join`, `distinct`, `orderBy`, віконні функції).
Дефолт **200** — історична константа, не повʼязана ні з обсягом даних, ні з розміром кластера.

Клітинки друкують `.rdd.getNumPartitions()` результату `groupBy` — 200, потім 16.
Spark UI → Jobs → stage після `Exchange` має рівно стільки ж tasks.

In [3]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", 200)
df.groupBy("PULocationID").agg(F.sum("fare_amount")).rdd.getNumPartitions()

200

In [4]:
spark.conf.set("spark.sql.shuffle.partitions", 16)
df.groupBy("PULocationID").agg(F.sum("fare_amount")).rdd.getNumPartitions()

16

Для ~2.96 млн рядків і 260 зон 200 партицій — надлишок: більшість task-ів опрацьовують 1–2 зони,
накладні на планування домінують. Замало партицій — гігантські партиції зі spill.
Орієнтир: 2–4 партиції на ядро. Здебільшого це питання знімає AQE — розділ 2.

## 2. `spark.sql.adaptive.enabled` — false vs true

AQE переоптимізовує план **під час виконання**: після shuffle враховує фактичні розміри партицій і зливає
дрібні. Клітинки друкують фінальну кількість партицій того самого `groupBy`: 200 → мала кількість
(тут 1: даних на кілька мегабайтів).

In [5]:
spark.conf.set("spark.sql.shuffle.partitions", 200)
spark.conf.set("spark.sql.adaptive.enabled", "false")
df.groupBy("PULocationID").agg(F.sum("fare_amount")).rdd.getNumPartitions()

200

In [6]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
df.groupBy("PULocationID").agg(F.sum("fare_amount")).rdd.getNumPartitions()

1

З увімкненим AQE `spark.sql.shuffle.partitions` можна ставити свідомо великим — надлишок зіллється.
У Spark UI → SQL план із AQE містить вузол `AQEShuffleRead` замість `Exchange`.

## 3. `spark.sql.adaptive.advisoryPartitionSizeInBytes` — 32 MB vs 2 MB

Параметр, яким AQE керує **розміром** партиції після coalesce. Запит із великим виходом (глобальний `orderBy`).
Клітинки пишуть результат і друкують кількість файлів = фінальна кількість партицій.

In [7]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "32m")
shutil.rmtree("/tmp/l15_adv", ignore_errors=True)
df.orderBy("fare_amount").write.mode("overwrite").parquet("/tmp/l15_adv")
len(glob.glob("/tmp/l15_adv/part-*.parquet"))

5

In [8]:
spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "2m")
shutil.rmtree("/tmp/l15_adv", ignore_errors=True)
df.orderBy("fare_amount").write.mode("overwrite").parquet("/tmp/l15_adv")
len(glob.glob("/tmp/l15_adv/part-*.parquet"))

45

## 4. `spark.sql.autoBroadcastJoinThreshold` — -1 vs 10 MB

`trips ⋈ zones` (265 рядків). `-1` вимикає авто-broadcast. Клітинки друкують компактний план;
тип join — у першому рядку:
- `-1` → **SortMergeJoin**: `Exchange` + `Sort` на кожній стороні (два shuffle-стейджі).
- `10m` → **BroadcastHashJoin**: `BroadcastExchange`, велика таблиця не шафлиться.

In [9]:
zones = (spark.read.parquet(ZONES)
         .select(F.col("LocationID").alias("zid"), F.col("Borough").alias("borough")))
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
df.join(zones, df.PULocationID == zones.zid).groupBy("borough").count().explain()

== Physical Plan ==
*(6) HashAggregate(keys=[borough#122], functions=[count(1)])
+- Exchange hashpartitioning(borough#122, 200), ENSURE_REQUIREMENTS, [plan_id=338]
   +- *(5) HashAggregate(keys=[borough#122], functions=[partial_count(1)])
      +- *(5) Project [borough#122]
         +- *(5) SortMergeJoin [cast(PULocationID#7 as bigint)], [zid#121L], Inner
            :- *(2) Sort [cast(PULocationID#7 as bigint) ASC NULLS FIRST], false, 0
            :  +- Exchange hashpartitioning(cast(PULocationID#7 as bigint), 200), ENSURE_REQUIREMENTS, [plan_id=319]
            :     +- *(1) Filter isnotnull(PULocationID#7)
            :        +- *(1) ColumnarToRow
            :           +- FileScan parquet [PULocationID#7] Batched: true, DataFilters: [isnotnull(PULocationID#7)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/data/so..., PartitionFilters: [], PushedFilters: [IsNotNull(PULocationID)], ReadSchema: struct<PULocatio

In [10]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10m")
df.join(zones, df.PULocationID == zones.zid).groupBy("borough").count().explain()

== Physical Plan ==
*(3) HashAggregate(keys=[borough#122], functions=[count(1)])
+- Exchange hashpartitioning(borough#122, 200), ENSURE_REQUIREMENTS, [plan_id=445]
   +- *(2) HashAggregate(keys=[borough#122], functions=[partial_count(1)])
      +- *(2) Project [borough#122]
         +- *(2) BroadcastHashJoin [cast(PULocationID#7 as bigint)], [zid#121L], Inner, BuildRight, false
            :- *(2) Filter isnotnull(PULocationID#7)
            :  +- *(2) ColumnarToRow
            :     +- FileScan parquet [PULocationID#7] Batched: true, DataFilters: [isnotnull(PULocationID#7)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/data/so..., PartitionFilters: [], PushedFilters: [IsNotNull(PULocationID)], ReadSchema: struct<PULocationID:int>
            +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, true]),false), [plan_id=439]
               +- *(1) Project [LocationID#117L AS zid#121L, Borough#118 AS

## 5. Spill: `spark.sql.shuffle.partitions` 1 vs 64 на важкому сортуванні

Глобальний `orderBy` потроєного датасету. При **1** партиції одна задача сортує ~8.9 млн рядків і не
вміщається у 900 MB heap. Клітинки друкують час. У Spark UI → Stages → sort-stage → таблиця Tasks унизу:
при 1 партиції колонки **Spill (memory)** / **Spill (disk)** ненульові (гігабайти); при 64 spill зникає.

In [11]:
wide = df.crossJoin(spark.range(3).toDF("dup"))   # ~8.9 млн рядків
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", 1)
t = time.time(); noop(wide.orderBy("fare_amount")); round(time.time() - t, 1)

13.5

In [12]:
spark.conf.set("spark.sql.shuffle.partitions", 64)
t = time.time(); noop(wide.orderBy("fare_amount")); round(time.time() - t, 1)

8.6

Порядок дій зі spill: спершу зменшення даних на партицію (фільтр раніше, менше колонок, більше партицій),
`spark.executor.memory` — останній крок.

## 6. Skew в агрегації + salting

Реальний PULocationID перекошений помірно. Синтетичний гострий перекіс: ключ `0` = ~65% рядків
(перша клітинка — розподіл).

Друга клітинка — `groupBy("k").agg(sort_array(collect_list(...)))`: reduce-задача ключа 0 сортує масив
на ~1.9 млн елементів, решта — крихітні. Spark UI → Stages → цей stage → таблиця Tasks унизу,
сортування за Duration: один рядок у ~10+ разів довший за решту.

Третя клітинка — salting (лише для **згортуваних** агрегацій sum/count/min/max): компактний план містить
**два** `Exchange` — часткова агрегація по `k+salt`, потім фінальна по `k`. Час менший.

In [13]:
skewed = df.withColumn(
    "k", F.when(F.rand(42) < 0.65, F.lit(0)).otherwise((F.rand(7) * 400 + 1).cast("int")))
skewed.groupBy("k").count().orderBy(F.desc("count")).show(5)

+---+-------+
|  k|  count|
+---+-------+
|  0|1927444|
|213|   2762|
|266|   2740|
|277|   2733|
|220|   2731|
+---+-------+
only showing top 5 rows


In [14]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", 24)
t = time.time()
noop(skewed.groupBy("k").agg(F.sort_array(F.collect_list("fare_amount"))))
round(time.time() - t, 1)

2.1

In [15]:
N = 24
salted = (skewed
    .withColumn("salt", (F.rand() * N).cast("int"))
    .groupBy("k", "salt").agg(F.sum("fare_amount").alias("partial"))
    .groupBy("k").agg(F.sum("partial").alias("total")))
salted.explain()
t = time.time(); noop(salted); round(time.time() - t, 1)

== Physical Plan ==
*(3) HashAggregate(keys=[k#176], functions=[sum(partial#342)])
+- Exchange hashpartitioning(k#176, 24), ENSURE_REQUIREMENTS, [plan_id=726]
   +- *(2) HashAggregate(keys=[k#176], functions=[partial_sum(partial#342)])
      +- *(2) HashAggregate(keys=[k#176, salt#341], functions=[sum(fare_amount#10)])
         +- Exchange hashpartitioning(k#176, salt#341, 24), ENSURE_REQUIREMENTS, [plan_id=721]
            +- *(1) HashAggregate(keys=[k#176, salt#341], functions=[partial_sum(fare_amount#10)])
               +- *(1) Project [fare_amount#10, k#176, cast((rand(8294501876357443683) * 24.0) as int) AS salt#341]
                  +- *(1) Project [fare_amount#10, CASE WHEN (rand(42) < 0.65) THEN 0 ELSE cast(((rand(7) * 400.0) + 1.0) as int) END AS k#176]
                     +- *(1) ColumnarToRow
                        +- FileScan parquet [fare_amount#10] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/illia/projects/UA_DATA-

1.4

## 7. `spark.sql.adaptive.skewJoin.enabled` — false vs true

Для **join-ів** перекіс лікує AQE автоматично: розбиває перекошену партицію на підзадачі.
`dim` тут із 12 рядками на ключ → join «розмножує» рядки, ключ 0 = ~1.9М×12 в одній reduce-задачі.
Клітинки друкують час; у Spark UI → Stages → join-stage → таблиця Tasks при `true` найдовша задача
падає до рівня решти.

In [16]:
dim = (spark.range(0, 401 * 12).toDF("j").withColumn("k", F.col("j") % 401).drop("j"))
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", 24)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
t = time.time()
noop(skewed.join(dim.hint("merge"), "k").selectExpr("k", "fare_amount"))
round(time.time() - t, 1)

4.1

In [17]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", 2)
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1m")
t = time.time()
noop(skewed.join(dim.hint("merge"), "k").selectExpr("k", "fare_amount"))
round(time.time() - t, 1)

2.1

## 8. UDF розриває оптимізований план

Python UDF — чорна скринька для Catalyst. Клітинки друкують компактний план + час:
- native вираз — суцільний codegen, жодного Python-вузла;
- `pandas_udf` — вузол `ArrowEvalPython` (дані пачками через Arrow);
- звичайна `@udf` — вузол `BatchEvalPython`, codegen обривається.

In [18]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
T = 40   # штучно важке обчислення на рядок

native = df.withColumn("y", sum((F.sin(F.col("fare_amount") / i) / i for i in range(1, T)), F.lit(0.0)))
native.explain()
t = time.time(); native.selectExpr("sum(y)").collect(); round(time.time() - t, 2)

== Physical Plan ==
*(1) Project [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, (((((((((((((((((((((((((((((((((((((((0.0 + (SIN((fare_amount#10 / 1.0)) / 1.0)) + (SIN((fare_amount#10 / 2.0)) / 2.0)) + (SIN((fare_amount#10 / 3.0)) / 3.0)) + (SIN((fare_amount#10 / 4.0)) / 4.0)) + (SIN((fare_amount#10 / 5.0)) / 5.0)) + (SIN((fare_amount#10 / 6.0)) / 6.0)) + (SIN((fare_amount#10 / 7.0)) / 7.0)) + (SIN((fare_amount#10 / 8.0)) / 8.0)) + (SIN((fare_amount#10 / 9.0)) / 9.0)) + (SIN((fare_amount#10 / 10.0)) / 10.0)) + (SIN((fare_amount#10 / 11.0)) / 11.0)) + (SIN((fare_amount#10 / 12.0)) / 12.0)) + (SIN((fare_amount#10 / 13.0)) / 13.0)) + (SIN((fare_amount#10 / 14.0)) / 14.0)) + (SIN((fare_amount#10 / 15.0

0.81

In [19]:
@F.pandas_udf(DoubleType())
def y_pandas(s: pd.Series) -> pd.Series:
    acc = np.zeros(len(s))
    for i in range(1, T):
        acc += np.sin(s.to_numpy() / i) / i
    return pd.Series(acc)

pud = df.withColumn("y", y_pandas("fare_amount"))
pud.explain()
t = time.time(); pud.selectExpr("sum(y)").collect(); round(time.time() - t, 2)

== Physical Plan ==
*(2) Project [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, pythonUDF0#393 AS y#392]
+- ArrowEvalPython [y_pandas(fare_amount#10)#391], [pythonUDF0#393], 200
   +- *(1) ColumnarToRow
      +- FileScan parquet [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/data/so..

3.82

In [20]:
@F.udf(DoubleType())
def y_py(x):
    return None if x is None else sum(math.sin(x / i) / i for i in range(1, T))

pyu = df.withColumn("y", y_py("fare_amount"))
pyu.explain()
t = time.time(); pyu.selectExpr("sum(y)").collect(); round(time.time() - t, 2)

== Physical Plan ==
*(2) Project [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, pythonUDF0#402 AS y#401]
+- BatchEvalPython [y_py(fare_amount#10)#400], [pythonUDF0#402]
   +- *(1) ColumnarToRow
      +- FileScan parquet [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/data/so..., Partit

8.72

## 9. `coalesce(1)` vs `repartition(1)` — пастка паралелізму

`coalesce(1)` — вузол `Coalesce`, **межі stage немає** → усе важке обчислення виконує **одна** задача.
`repartition(1)` — вузол `Exchange`, **межа stage є** → обчислення паралельне на 4 ядрах, у один потік
звужується лише запис. Клітинки друкують компактний план + час.

In [21]:
heavy = (spark.read.parquet(SRC).crossJoin(spark.range(2).toDF("d"))
         .filter(F.col("fare_amount") > 0)
         .withColumn("z", sum((F.sin(F.col("fare_amount") / i) / i for i in range(1, 80)), F.lit(0.0))))
heavy.coalesce(1).explain()
t = time.time(); noop(heavy.coalesce(1)); round(time.time() - t, 1)

== Physical Plan ==
Coalesce 1
+- *(3) Project [VendorID#409, tpep_pickup_datetime#410, tpep_dropoff_datetime#411, passenger_count#412L, trip_distance#413, RatecodeID#414L, store_and_fwd_flag#415, PULocationID#416, DOLocationID#417, payment_type#418L, fare_amount#419, extra#420, mta_tax#421, tip_amount#422, tolls_amount#423, improvement_surcharge#424, total_amount#425, congestion_surcharge#426, Airport_fee#427, d#429L, (((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((0.0 + (SIN((fare_amount#419 / 1.0)) / 1.0)) + (SIN((fare_amount#419 / 2.0)) / 2.0)) + (SIN((fare_amount#419 / 3.0)) / 3.0)) + (SIN((fare_amount#419 / 4.0)) / 4.0)) + (SIN((fare_amount#419 / 5.0)) / 5.0)) + (SIN((fare_amount#419 / 6.0)) / 6.0)) + (SIN((fare_amount#419 / 7.0)) / 7.0)) + (SIN((fare_amount#419 / 8.0)) / 8.0)) + (SIN((fare_amount#419 / 9.0)) / 9.0)) + (SIN((fare_amount#419 / 10.0)) / 10.0)) + (SIN((fare_amount#419 / 11.0)) / 11.0)) + (SIN((fare_amount#419 / 12.0)) / 12.0)) + (SIN((

51.4

In [22]:
heavy.repartition(1).explain()
t = time.time(); noop(heavy.repartition(1)); round(time.time() - t, 1)

== Physical Plan ==
Exchange SinglePartition, REPARTITION_BY_NUM, [plan_id=1439]
+- *(3) Project [VendorID#409, tpep_pickup_datetime#410, tpep_dropoff_datetime#411, passenger_count#412L, trip_distance#413, RatecodeID#414L, store_and_fwd_flag#415, PULocationID#416, DOLocationID#417, payment_type#418L, fare_amount#419, extra#420, mta_tax#421, tip_amount#422, tolls_amount#423, improvement_surcharge#424, total_amount#425, congestion_surcharge#426, Airport_fee#427, d#429L, (((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((0.0 + (SIN((fare_amount#419 / 1.0)) / 1.0)) + (SIN((fare_amount#419 / 2.0)) / 2.0)) + (SIN((fare_amount#419 / 3.0)) / 3.0)) + (SIN((fare_amount#419 / 4.0)) / 4.0)) + (SIN((fare_amount#419 / 5.0)) / 5.0)) + (SIN((fare_amount#419 / 6.0)) / 6.0)) + (SIN((fare_amount#419 / 7.0)) / 7.0)) + (SIN((fare_amount#419 / 8.0)) / 8.0)) + (SIN((fare_amount#419 / 9.0)) / 9.0)) + (SIN((fare_amount#419 / 10.0)) / 10.0)) + (SIN((fare_amount#419 / 11.0)) / 11.0)) 

28.8

## 10. Кеш і повторне обчислення (lazy evaluation)

Кожна action переобчислює ланцюг з нуля. Перша клітинка — **дві** дії над одним DataFrame без кешу
(Spark UI → Jobs: два скани джерела). Друга — те саме з `cache()`: компактний план другої дії показує
`InMemoryTableScan` замість `Scan parquet`, Spark UI → Storage — закешовані партиції. Час друкується.

In [23]:
spark.catalog.clearCache()
heavy_c = sum((F.sin(F.col("fare_amount") / i) / i for i in range(1, 80)), F.lit(0.0))
clean = (spark.read.parquet(SRC).select("fare_amount")
         .filter(F.col("fare_amount") > 0).withColumn("h", heavy_c))
t = time.time()
clean.agg(F.sum("h")).collect(); clean.agg(F.avg("h")).collect()
round(time.time() - t, 2)

2.4

In [24]:
clean.cache(); clean.count()          # матеріалізація
t = time.time()
clean.agg(F.sum("h")).collect(); clean.agg(F.avg("h")).collect()
clean.agg(F.sum("h")).explain()       # InMemoryTableScan
round(time.time() - t, 2)

== Physical Plan ==
*(2) HashAggregate(keys=[], functions=[sum(h#452)])
+- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=1715]
   +- *(1) HashAggregate(keys=[], functions=[partial_sum(h#452)])
      +- InMemoryTableScan [h#452]
            +- InMemoryRelation [fare_amount#442, h#452], StorageLevel(disk, memory, deserialized, 1 replicas)
                  +- *(1) Project [fare_amount#442, (((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((((0.0 + (SIN((fare_amount#442 / 1.0)) / 1.0)) + (SIN((fare_amount#442 / 2.0)) / 2.0)) + (SIN((fare_amount#442 / 3.0)) / 3.0)) + (SIN((fare_amount#442 / 4.0)) / 4.0)) + (SIN((fare_amount#442 / 5.0)) / 5.0)) + (SIN((fare_amount#442 / 6.0)) / 6.0)) + (SIN((fare_amount#442 / 7.0)) / 7.0)) + (SIN((fare_amount#442 / 8.0)) / 8.0)) + (SIN((fare_amount#442 / 9.0)) / 9.0)) + (SIN((fare_amount#442 / 10.0)) / 10.0)) + (SIN((fare_amount#442 / 11.0)) / 11.0)) + (SIN((fare_amount#442 / 12.0)) / 12.0)) + (SIN((fare_amount#442 / 13.0

0.52

## 11. Розмір вихідних файлів — 200 партицій vs `coalesce(16)`

Багато shuffle-партицій → багато дрібних файлів (metadata на кожен, окрема задача на читанні).
Клітинки пишуть датасет і друкують `(кількість файлів, середній розмір KB)`.

In [25]:
enriched = spark.read.parquet(SRC).filter(F.col("fare_amount") > 0).repartition(200, "PULocationID")
shutil.rmtree("/tmp/l15_files", ignore_errors=True)
enriched.write.mode("overwrite").parquet("/tmp/l15_files")
fs = glob.glob("/tmp/l15_files/part-*.parquet")
(len(fs), round(sum(os.path.getsize(f) for f in fs) / len(fs) / 1024))

(145, 425)

In [26]:
shutil.rmtree("/tmp/l15_files", ignore_errors=True)
enriched.coalesce(16).write.mode("overwrite").parquet("/tmp/l15_files")
fs = glob.glob("/tmp/l15_files/part-*.parquet")
(len(fs), round(sum(os.path.getsize(f) for f in fs) / len(fs) / 1024))

(16, 3815)

## 12. Модель памʼяті executor-а

JVM-heap executor-а: `Reserved 300 MB` + `User Memory (~40%)` + `Unified = Execution + Storage`
(`spark.memory.fraction` = 0.6; межа рухома, execution витісняє storage).
Python-worker живе поза JVM heap → покривається `spark.executor.memoryOverhead`.
Контейнер = `executor.memory + memoryOverhead (+ offHeap)`; перевищення overhead → процес завершується
OOM killer-ом (**exit 137**), без Spark-стектрейсу — типова причина падіння PySpark на `pandas_udf` / `toPandas()`.

Spark UI → Executors: `Storage Memory`, `Peak Execution / Storage Memory`, `GC Time` (> 10% Task Time — проблема).

| Симптом | Регіон | Важіль |
|---|---|---|
| Spill (memory/disk) > 0 | брак **Execution** | більше партицій / менше `executor.cores` / `memory.fraction`↑ |
| `OutOfMemoryError` у JVM | User Memory / `collect()` / завелика broadcast | `driver.memory`↑, `autoBroadcastJoinThreshold`↓ |
| контейнер killed, **exit 137** | overhead (Python / Arrow) | `memoryOverhead`↑ / `pyspark.memory`↑ |
| GC Time > 10% Task Time | heap замалий / забагато cache | `executor.memory`↑ / прибрати cache / `MEMORY_ONLY_SER` |

In [27]:
for k in ["spark.driver.memory", "spark.executor.memory", "spark.executor.memoryOverhead",
          "spark.memory.fraction", "spark.memory.storageFraction", "spark.serializer"]:
    try:
        print(f"{k:34} = {spark.conf.get(k)}")
    except Exception:
        print(f"{k:34} = <дефолт, не заданий явно>")

spark.driver.memory                = 900m
spark.executor.memory              = <дефолт, не заданий явно>
spark.executor.memoryOverhead      = <дефолт, не заданий явно>
spark.memory.fraction              = <дефолт, не заданий явно>
spark.memory.storageFraction       = <дефолт, не заданий явно>
spark.serializer                   = org.apache.spark.serializer.KryoSerializer


## Чек-лист (у порядку спадання ефекту)

1. Читати менше — фільтр і `select` якнайраніше.
2. Прибрати зайві shuffle-и, зменшити обсяг того, що лишився.
3. `broadcast` там, де одна сторона мала.
4. Полагодити skew — фільтром службових значень, AQE skewJoin або salting.
5. Виправити кількість партицій під дані й кластер (або довіритись AQE).
6. Прибрати UDF / замінити на нативні вирази чи `pandas_udf`.
7. Кешувати те, що справді читається кількома діями.
8. Полагодити розмір вихідних файлів.
9. І лише потім — `spark.executor.memory` та кількість executor-ів.

Event log цього прогону — у `lesson-15-spark-optimization/code/spark-events/` (Spark History Server).
`# spark.stop()` — виклик не потрібен, поки Spark UI має лишатися доступною.